# AGEMR：1-df SNP×age directional enrichment

**已确认的定向：** 每个 phenotype 按自己的 pooled GWAS 主效应升高等位基因定向。BMI 用 BMI 主效应，SBP 用 SBP 主效应，DBP 用 DBP 主效应。

在已有 linear run 上，使用 All、P_main < 10⁻³、10⁻⁵、5×10⁻⁸ 四组。主要方向分析包含该组的**全部有效交互结果**，不要求单 SNP 交互显著；交互 P<0.05、面板 Bonferroni 子集另作描述性补充。

**只读取已有结果，不需要 PLINK、不重新拟合、不下载 controlled data。** 所有新结果写入同一个 workspace 的新子目录。可在新 kernel Run All；默认寻找最近一份同时包含 linear 与 pooled-main 的完整 run，也可指定 LINEAR_RUN。

**范围：** 使用 1-df 线性年龄交互系数；原来的 4-df omnibus P/F 没有正负号，不能给它强加一个方向。4-df 方向需事先定义年龄对比并获得相应系数/协方差，这份 notebook 不替代该分析。

## 方向定义与统计假设

先将两个模型的效应都转换为 **ALT dosage** 定义，再计算

$$d_j=\operatorname{sign}(\hat\beta_{\mathrm{pooled},j}^{ALT})\,\hat\theta_j^{ALT},\qquad
Z_{dir,j}=d_j/SE(\hat\theta_j).$$

- $d_j>0$：性状升高等位基因的效应随年龄变得更正（与 pooled 主效应同向）。
- $d_j<0$：性状升高等位基因的效应随年龄变得更负（与 pooled 主效应反向）。
- $d_j$ 的单位是 phenotype unit / allele / 10 年。

这些分别常被描述为 intensification / attenuation，但只凭斜率和 pooled 符号，不能证明整个年龄区间的绝对效应都单调增强/减弱；存在 crossing 时尤其需要看实际效应曲线。主效应接近零时定向不稳定，All 组仅作总体参照，不把噪声方向当成可靠生物学定向。

**主要报告的集合统计量**是 $S=\sum_j\operatorname{sign}(d_j)=N_+-N_-$，采用双侧检验，不在看到数据后选单侧方向。

- **按染色体块翻转的参考 P：** 把同一染色体所有定向斜率整体乘 +1 或 −1，再比较 |S|。这样保留染色体内的符号相关模式。非零块≤16 时枚举全部翻转，否则固定种子 Monte Carlo（49,999 次，P 用 +1 修正）。要求至少 10 个非零斜率 SNP、至少 10 条有数据的染色体；不足时只给计数。它仍依赖染色体之间可近似独立、零假设下块向量中心对称、定向/筛选可视为合适条件等假设，**作为探索性参考和 LD 敏感性分析，不声称已完成全数据的经验校准**。
- **补充 binomial sign-test P：** 正负概率 1/2 的双侧精确检验；把 SNP 视为独立，是明确标注的 iid 参考，不能忽略剩余 LD。
- **补充 signed-Stouffer P：** $\sum Z_{dir}/\sqrt{m}$，是独立、近似标准正态交互统计量下的参考，可能由少量强信号驱动。与方向计数检验不是同一个零假设。

预先把 3 traits × 4 主效应组的 **12 个“全部交互 P”块翻转检验**定义为一族，报告固定 12 倍 Bonferroni；嵌套组仍计入。交互显著子集和其他统计量不纳入这一主族，均标为补充探索。Bonferroni 不能修正原始 P 本身的模型/校准问题。

方法背景：[年龄交互方向解释](https://www.nature.com/articles/s43587-026-01232-w)、[marginal screening 的条件与理论](https://pmc.ncbi.nlm.nih.gov/articles/PMC3629859/)、[SciPy binomtest](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html)。这些文献不构成本 notebook 的全部 LD/误差假设已在 AoU 得到满足的证据。来源核查于 2026-10-28。

In [ ]:
from pathlib import Path
import json, hashlib, uuid
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from IPython.display import display

LINEAR_RUN = None
B1_ROOT = Path.home() / "workspace/agemr-data/final_cohort_v1/B1_age_varying"
TRAITS = ("BMI", "SBP", "DBP")
GROUPS = [("All",None),("P_main < 1e-3",1e-3),("P_main < 1e-5",1e-5),("P_main < 5e-8",5e-8)]
KEY = ["#CHROM","POS","ID","REF","ALT"]
MIN_NONZERO_SNPS = 10
MIN_CHROMOSOMES = 10
N_SIGN_FLIPS = 49999
BASE_SEED = 20261028
ALPHA = 0.05
FAMILY_SIZE = len(TRAITS)*len(GROUPS)

def require(ok,msg):
    if not ok:raise RuntimeError(msg)

if LINEAR_RUN is None:
    choices=[]
    for marker in (B1_ROOT / "B1_linear_age").glob("*/RUN_COMPLETE.json"):
        meta=json.loads(marker.read_text())
        if meta.get("status")=="complete" and all((marker.parent/f"{t}_linear_age_all.tsv").is_file() and
           (marker.parent/"main_effect_stratified_proportions"/f"{t}_pooled_main.tsv").is_file() for t in TRAITS):
            choices.append((meta.get("finished_utc",""),marker.parent))
    require(bool(choices),"No completed linear + pooled-main run found; set LINEAR_RUN.")
    LINEAR_RUN=max(choices,key=lambda z:z[0])[1]
else:
    LINEAR_RUN=Path(LINEAR_RUN).expanduser()
meta=json.loads((LINEAR_RUN / "RUN_COMPLETE.json").read_text())
require(meta.get("status")=="complete","Selected linear run is incomplete")
M=int(meta["panel_variants"])
SNP_BONF=ALPHA/M
stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:6]
OUT=LINEAR_RUN / "directional_enrichment_1df" / stamp
for folder in (OUT,OUT/"qc",OUT/"groups",OUT/"figures"):
    folder.mkdir(parents=True,exist_ok=True)
print("Using existing run:",LINEAR_RUN)
print("New outputs:",OUT)
print(f"SNP Bonferroni={SNP_BONF:.6e}; primary directional family={FAMILY_SIZE}")

sources=[]
def sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""):h.update(b)
    return h.hexdigest()

def read_result(path):
    x=pd.read_csv(path,sep="\t",dtype=str,keep_default_na=False)
    require(set(KEY).issubset(x),f"Unexpected schema: {path}")
    x["#CHROM"]=x["#CHROM"].str.replace(r"^chr","",regex=True)
    x["POS"]=pd.to_numeric(x.POS,errors="raise").astype(int)
    require(not x.duplicated(KEY).any(),f"Duplicate variant keys: {path}")
    for c in ("P","BETA","SE","OBS_CT"):
        require(c in x,f"Missing {c}: {path}")
        x[c]=pd.to_numeric(x[c],errors="coerce")
    sources.append({"path":str(path),"bytes":Path(path).stat().st_size,"sha256":sha(path)})
    return x

def truth(x):return x.astype(str).str.lower().isin(["true","1"])

def prepare_directional(interaction,main):
    require(interaction.TEST.eq("ADDxAGE10").all(),"Expected 1-df ADDxAGE10 results")
    require(main.TEST.eq("ADD").all(),"Expected no-interaction pooled ADD results")
    use=KEY+[c for c in ["A1","BETA","SE","P","OBS_CT","ERRCODE","VALID","rsid"] if c in interaction]
    x=interaction[use].rename(columns={c:c+"_int" for c in use if c not in KEY})
    y=main[KEY+["A1","BETA","SE","P","OBS_CT","ERRCODE"]].rename(columns={c:c+"_main" for c in ["A1","BETA","SE","P","OBS_CT","ERRCODE"]})
    x=x.merge(y,on=KEY,how="outer",validate="one_to_one",indicator="key_match")
    x["valid_int"]=(truth(x["VALID_int"]) & x["ERRCODE_int"].eq(".") &
        x["P_int"].between(0,1) & np.isfinite(x["BETA_int"]) & np.isfinite(x["SE_int"]) & x["SE_int"].gt(0))
    x["valid_main"]=(x["ERRCODE_main"].eq(".") & x["P_main"].between(0,1) &
        np.isfinite(x["BETA_main"]) & np.isfinite(x["SE_main"]) & x["SE_main"].gt(0))
    for suffix in ("int","main"):
        a1=x[f"A1_{suffix}"]
        x[f"allele_ok_{suffix}"]=a1.eq(x.ALT) | a1.eq(x.REF)
        x[f"to_ALT_{suffix}"]=np.where(a1.eq(x.ALT),1.0,np.where(a1.eq(x.REF),-1.0,np.nan))
        x[f"BETA_ALT_{suffix}"]=x[f"BETA_{suffix}"]*x[f"to_ALT_{suffix}"]
    x["same_OBS_CT"]=x.OBS_CT_main.eq(x.OBS_CT_int) & x.OBS_CT_int.gt(0)
    x["main_direction_nonzero"]=x.BETA_ALT_main.ne(0) & np.isfinite(x.BETA_ALT_main)
    x["eligible"]=(x.key_match.eq("both") & x.valid_int & x.valid_main & x.same_OBS_CT &
                   x.allele_ok_int & x.allele_ok_main & x.main_direction_nonzero)
    x["trait_increasing_allele"]=np.where(x.BETA_ALT_main.gt(0),x.ALT,np.where(x.BETA_ALT_main.lt(0),x.REF,"undefined"))
    x["orientation_sign_ALT"]=np.sign(x.BETA_ALT_main)
    x["BETA_dir_per_10y"]=x.orientation_sign_ALT*x.BETA_ALT_int
    x["SE_dir_per_10y"]=x.SE_int
    x["Z_dir_approx"]=x.BETA_dir_per_10y/x.SE_int
    x["direction"]=np.select([~x.eligible,x.BETA_dir_per_10y.gt(0),x.BETA_dir_per_10y.lt(0)],
                             ["excluded","positive_aligned_slope","negative_aligned_slope"],default="zero_slope")
    return x

aligned={}
qc=[]
for trait in TRAITS:
    interaction=read_result(LINEAR_RUN/f"{trait}_linear_age_all.tsv")
    main=read_result(LINEAR_RUN/"main_effect_stratified_proportions"/f"{trait}_pooled_main.tsv")
    x=prepare_directional(interaction,main)
    require(len(x)==M and x.key_match.eq("both").all(),f"{trait}: incomplete/different panel; inspect keys before analysis")
    x["trait"]=trait
    x.to_csv(OUT/f"{trait}_allele_aligned_directional.tsv",sep="\t",index=False)
    x.loc[~x.eligible].to_csv(OUT/"qc"/f"{trait}_excluded_variants.tsv",sep="\t",index=False)
    qc.append({"trait":trait,"N_panel":len(x),"N_eligible":int(x.eligible.sum()),
               "N_invalid_int":int((~x.valid_int).sum()),"N_invalid_main":int((~x.valid_main).sum()),
               "N_allele_unresolved":int((~(x.allele_ok_int & x.allele_ok_main)).sum()),
               "N_OBS_CT_mismatch":int((~x.same_OBS_CT).sum()),
               "N_zero_main_direction":int((x.BETA_ALT_main==0).sum()),
               "N_A1_different_between_models":int(x.A1_int.ne(x.A1_main).sum())})
    aligned[trait]=x
qc=pd.DataFrame(qc)
qc.to_csv(OUT/"qc/allele_orientation_summary.tsv",sep="\t",index=False)
display(qc)

## 方向检验及染色体块翻转

主要方向性假设是预先指定的双侧检验，不把观察后更显著的一侧当作预设。整条染色体翻转只是一种块级零假设参考，不能处理跨染色体依赖，也不能消除模型误设、亲缘结构、选择或 phenotype scale 的影响。

同样本 pooled 主效应筛选**并不自动意味着无效**：嵌套规范模型下有相应独立性理论。但这些条件、误差结构以及定向后联合分布需要审查，不能仅凭中心化年龄或已有 pruning 标签宣布全部满足。弱主效应定向的不确定性仍然存在。

In [ ]:
def block_flip_p(block_scores,seed,draws=N_SIGN_FLIPS):
    scores=np.asarray(block_scores,dtype=np.int64)
    scores=scores[scores!=0]
    k=len(scores)
    observed=abs(int(scores.sum()))
    if k==0:return 1.0,"all_block_scores_zero",0
    if k<=16:
        numbers=np.arange(1<<k,dtype=np.uint32)
        signs=1-2*((numbers[:,None]>>np.arange(k,dtype=np.uint32))&1).astype(np.int8)
        null=signs@scores
        return float(np.mean(np.abs(null)>=observed)),"exact_block_enumeration",len(numbers)
    rng=np.random.default_rng(seed)
    exceed=0
    completed=0
    while completed<draws:
        batch=min(4096,draws-completed)
        signs=(2*rng.integers(0,2,size=(batch,k),dtype=np.int8)-1)
        exceed+=int(np.sum(np.abs(signs@scores)>=observed))
        completed+=batch
    return (exceed+1)/(draws+1),"Monte_Carlo_block_sign_flip",draws

def directional_summary(g,trait,group,set_label):
    n=len(g)
    sign=np.sign(g.BETA_dir_per_10y.to_numpy(dtype=float)).astype(np.int64)
    npos=int((sign>0).sum());nneg=int((sign<0).sum());nzero=n-npos-nneg
    nonzero=npos+nneg
    nz=g.loc[g.BETA_dir_per_10y.ne(0)]
    chr_scores=nz.assign(_sgn=np.sign(nz.BETA_dir_per_10y).astype(int)).groupby("#CHROM")["_sgn"].sum()
    nchrom=len(chr_scores)
    row={"trait":trait,"main_group":group,"interaction_set":set_label,"N":n,
         "N_positive":npos,"N_negative":nneg,"N_zero_slope":nzero,"N_nonzero":nonzero,
         "N_chromosomes_nonzero_slopes":nchrom,"N_nonzero_chromosome_scores":int(chr_scores.ne(0).sum()),
         "negative_pct":100*nneg/nonzero if nonzero else np.nan,
         "positive_pct":100*npos/nonzero if nonzero else np.nan,"sign_excess_positive_minus_negative":npos-nneg,
         "mean_BETA_dir_per_10y":g.BETA_dir_per_10y.mean() if n else np.nan,
         "median_BETA_dir_per_10y":g.BETA_dir_per_10y.median() if n else np.nan,
         "mean_Z_dir_approx":g.Z_dir_approx.mean() if n else np.nan,
         "P_sign_binomial_iid_reference":np.nan,"Z_Stouffer_iid_reference":np.nan,
         "P_Stouffer_iid_reference":np.nan,"P_chr_block_sign_reference":np.nan,
         "block_method":"not_run","n_null_flips":0,"status":"too_few_nonzero_SNPs",
         "calibration_status":"exploratory_assumption_dependent"}
    if nonzero>=MIN_NONZERO_SNPS:
        row["P_sign_binomial_iid_reference"]=float(stats.binomtest(nneg,nonzero,.5,alternative="two-sided").pvalue)
        z=g.Z_dir_approx.sum()/np.sqrt(n)
        row["Z_Stouffer_iid_reference"]=z
        row["P_Stouffer_iid_reference"]=float(2*stats.norm.sf(abs(z)))
        row["status"]="too_few_chromosomes_for_block_test"
        if nchrom>=MIN_CHROMOSOMES:
            tag=f"{trait}|{group}|{set_label}".encode()
            seed=BASE_SEED ^ int(hashlib.sha256(tag).hexdigest()[:8],16)
            p,method,B=block_flip_p(chr_scores.to_numpy(),seed)
            row.update(P_chr_block_sign_reference=p,block_method=method,n_null_flips=B,status="reference_tests_computed")
    return row

rows=[]
leave_chr=[]
for trait in TRAITS:
    x=aligned[trait]
    for group_index,(label,cut) in enumerate(GROUPS):
        selected=pd.Series(True,index=x.index) if cut is None else x.P_main.between(0,1) & x.P_main.lt(cut)
        base=x.loc[selected & x.eligible].copy()
        for set_label,mask in [("all_interaction_P",pd.Series(True,index=base.index)),
                               ("interaction_P_lt_005",base.P_int.lt(ALPHA)),
                               ("interaction_panel_Bonf",base.P_int.lt(SNP_BONF))]:
            g=base.loc[mask].copy()
            row=directional_summary(g,trait,label,set_label)
            row["N_selected_by_main_P"]=int(selected.sum())
            row["N_excluded_alignment_or_QC"]=int((selected & ~x.eligible).sum())
            row["panel_Bonf_threshold"]=SNP_BONF
            rows.append(row)
            tag=("All","Pmain_lt_1e3","Pmain_lt_1e5","Pmain_lt_5e8")[group_index]
            g.to_csv(OUT/"groups"/f"{trait}_{tag}_{set_label}.tsv",sep="\t",index=False)
            if set_label=="all_interaction_P":
                for chrom in sorted(g["#CHROM"].unique(),key=int):
                    kept=g.loc[g["#CHROM"].ne(chrom)]
                    signs=np.sign(kept.BETA_dir_per_10y)
                    nonzero=int(signs.ne(0).sum())
                    leave_chr.append({"trait":trait,"main_group":label,"omitted_chr":chrom,"N_remaining":len(kept),
                        "negative_pct_remaining":100*float(signs.lt(0).sum())/nonzero if nonzero else np.nan,
                        "sign_excess_remaining":int(signs.gt(0).sum()-signs.lt(0).sum()),
                        "mean_Z_dir_remaining":kept.Z_dir_approx.mean() if len(kept) else np.nan})
summary=pd.DataFrame(rows)
primary=summary.loc[summary.interaction_set.eq("all_interaction_P")].copy()
require(len(primary)==FAMILY_SIZE,"Primary family size mismatch")
primary["P_chr_block_Bonf12_reference"]=np.minimum(primary.P_chr_block_sign_reference*FAMILY_SIZE,1.0)
primary["primary_family_size"]=FAMILY_SIZE
summary.to_csv(OUT/"directional_summary_all_interaction_sets.tsv",sep="\t",index=False)
primary.to_csv(OUT/"directional_primary_all_interaction_P.tsv",sep="\t",index=False)
pd.DataFrame(leave_chr).to_csv(OUT/"qc/leave_one_chromosome_out.tsv",sep="\t",index=False)

show=primary[["trait","main_group","N","N_positive","N_negative","negative_pct",
              "N_chromosomes_nonzero_slopes","P_sign_binomial_iid_reference",
              "P_chr_block_sign_reference","P_chr_block_Bonf12_reference",
              "Z_Stouffer_iid_reference","P_Stouffer_iid_reference","status"]].copy()
show["negative_pct"]=show.negative_pct.round(2)
show["Z_Stouffer_iid_reference"]=show.Z_Stouffer_iid_reference.round(3)
for c in show:
    if c.startswith("P_"):show[c]=show[c].map(lambda v:f"{v:.3e}" if pd.notna(v) else "NA")
display(show)
print("Interaction-significant subsets are exploratory, saved separately:")
display(summary.loc[~summary.interaction_set.eq("all_interaction_P"),
        ["trait","main_group","interaction_set","N","N_positive","N_negative","negative_pct","status"]])

In [ ]:
for trait in TRAITS:
    z=primary.loc[primary.trait.eq(trait)].set_index("main_group").loc[[name for name,_ in GROUPS]]
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    pos=np.arange(len(z))
    axes[0].bar(pos,z.negative_pct,color="#d7853a",label="Negative aligned slope")
    axes[0].bar(pos,z.positive_pct,bottom=z.negative_pct,color="#347ca7",label="Positive aligned slope")
    axes[0].axhline(50,color="black",ls="--",lw=1)
    labels=[f"{group}\n(n={int(n)})" for group,n in zip(z.index,z.N_nonzero)]
    axes[0].set_xticks(pos,labels)
    axes[0].set(ylim=(0,105),ylabel="Nonzero slopes (%)",title="Direction relative to own pooled main effect")
    axes[0].legend(fontsize=8,loc="upper center",bbox_to_anchor=(.5,-.20),ncol=2)
    x=aligned[trait]
    for name,cut,color in [("All",None,"#347ca7"),("P_main < 1e-3",1e-3,"#d7853a")]:
        g=x.loc[x.eligible & (pd.Series(True,index=x.index) if cut is None else x.P_main.lt(cut))]
        if len(g):
            capped=int(g.Z_dir_approx.abs().gt(8).sum())
            axes[1].hist(np.clip(g.Z_dir_approx,-8,8),bins=np.linspace(-8,8,41),density=True,
                         histtype="step",lw=1.6,color=color,label=f"{name}, n={len(g)}, capped={capped}")
    axes[1].axvline(0,color="black",ls="--",lw=1)
    axes[1].set(xlabel="Aligned interaction beta / SE (display capped at ±8)",ylabel="Density",
                title="Signed interaction statistics; all P values included")
    axes[1].legend(fontsize=8)
    fig.suptitle(f"{trait}: exploratory directional enrichment",fontsize=12)
    fig.tight_layout()
    fig.savefig(OUT/"figures"/f"{trait}_directional_enrichment.png",dpi=200,bbox_inches="tight")
    fig.savefig(OUT/"figures"/f"{trait}_directional_enrichment.pdf",bbox_inches="tight")
    plt.show();plt.close(fig)

pd.DataFrame(sources).to_csv(OUT/"qc/source_files.tsv",sep="\t",index=False)
(OUT/"DIRECTIONAL_COMPLETE.json").write_text(json.dumps({"status":"complete","linear_run":str(LINEAR_RUN),
    "created_utc":datetime.now(timezone.utc).isoformat(),"orientation":"own_phenotype_pooled_trait_increasing_allele",
    "age_interaction":"1df linear; nuisance retains complete age spline","groups":GROUPS,
    "primary_statistic":"N_positive_minus_N_negative","primary_reference_calibration":"chromosome_block_sign_flip",
    "reference_assumptions":"independent chromosome blocks; appropriate zero-null sign symmetry conditional on orientation/selection; valid interaction model",
    "inference_status":"exploratory_not_full_empirical_calibration","primary_family_size":FAMILY_SIZE,
    "min_nonzero_SNPs":MIN_NONZERO_SNPS,"min_chromosomes":MIN_CHROMOSOMES,
    "mc_sign_flips":N_SIGN_FLIPS,"seed":BASE_SEED},indent=2,ensure_ascii=False)+"\n")
print("Finished; all outputs remain in:",OUT)

## 看哪些结果

- `directional_primary_all_interaction_P.tsv`：主表，每个 trait × 主效应组的正/负数量、负向比例、参考 P 和 12 项 Bonferroni。
- `directional_summary_all_interaction_sets.tsv`：再加入 nominal / 面板 Bonferroni 交互子集的补充结果。
- `*_allele_aligned_directional.tsv`、`groups/`：每个 SNP 的原始 A1、转 ALT 的因子、主效应升高等位基因、定向后的斜率及 SE；用于人工复核。
- `qc/leave_one_chromosome_out.tsv`：删除单条染色体后方向模式是否变化。
- `figures/`：各 trait 的正负比例和 signed-statistic 分布图。

例如负向比例明显高于 50%，可描述为“按主效应升高等位基因对齐后，负的年龄斜率较多”。参考 P 支持程度取决于相应 LD、误差和筛选假设；不要直接称为已经证明纵向衰老导致遗传效应减弱。横断面 age interaction 也可能反映 cohort、选择、药物或 phenotype scale。

极小子集仅给数量。不要把 SBP/DBP 的 1 个强主效应 SNP 当成足以支持整体方向偏倚的证据。All 组的弱主效应方向不稳定，需结合更强主效应组和 leave-one-chromosome-out 一起看。

**验证范围：** 交付版本无真实 cohort 输出；使用合成 workspace 验证全部运行、等位基因翻转不变性、共同样本数检查、正负/零方向、严格筛选阈值、块翻转枚举与 Monte Carlo、空/极小子集。运行后再解释 AoU 结果。